# Remapping ERA5 Precipitation Data to HEALPix Grid

This notebook demonstrates the workflow for remapping ECMWF ERA5 mean precipitation rate data from a regular latitude-longitude grid to a HEALPix grid using Delaunay-based weights. The process includes data loading, time flattening, weight computation, and visualization on a global map projection.

### Function imports

In [1]:
import healpix as hp
import numpy as np
import xarray as xr
import uxarray as ux

import easygems.healpix as egh
import easygems.remap as egr

import cartopy.crs as ccrs
import cartopy.feature as cf
import matplotlib.pyplot as plt

### (Optional) Using Dask to parallize tasks

In [2]:
import dask 
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from dask.distributed import performance_report

rda_scratch = '/glade/derecho/scratch/khirata/'

cluster = PBSCluster(
    job_name = 'dask-wk24-hpc',
    cores = 8,
    memory = '256GiB',
    local_directory = rda_scratch+'/dask/spill',
    log_directory = rda_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=8:mem=256GB',
    queue = 'casper',
    walltime = '01:00:00',
    #interface = 'ib0'
    interface = 'ext'
)
# cluster = PBSCluster(
#     job_name = 'dask-wk24-hpc',
#     cores = 32,
#     memory = '1024GiB',
#     local_directory = rda_scratch+'/dask/spill',
#     log_directory = rda_scratch + '/dask/logs/',
#     resource_spec = 'select=1:ncpus=32:mem=1024GB',
#     queue = 'casper',
#     walltime = '12:00:00',
#     #interface = 'ib0'
#     interface = 'ext'
# )

cluster.scale(2) # large number may result in an error
# cluster.adapt(minimum=2, maximum=32)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.177:45057,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


Task exception was never retrieved
future: <Task finished name='Task-431535' coro=<Client._gather.<locals>.wait() done, defined at /glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/distributed/client.py:2371> exception=AllExit()>
Traceback (most recent call last):
  File "/glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/distributed/client.py", line 2380, in wait
    raise AllExit()
distributed.client.AllExit


### Read in ERA5 data into Xarray Dataset

In [28]:
import pandas as pd
import glob

start_date = '2018-12-01'
# start_date = '2021-01-01'
end_date = '2021-12-31'

vars_to_read = ['mcpr', 'mlspr']  # Mean convective precipitation rate and Mean large-scale precipitation rate

months = pd.date_range(start=start_date, end=end_date, freq='MS')
fns = []
for var in vars_to_read:
    for dt in months:
        yyyy = dt.year
        mm = dt.month
        pattern = f'/glade/campaign/collections/rda/data/d633000/e5.oper.fc.sfc.meanflux/{yyyy}{mm:02d}/e5.oper.fc.sfc.meanflux.*_{var}.*.nc'
        fns.extend(sorted(glob.glob(pattern)))

# Remove duplicates in case files overlap between variables
fns = sorted(set(fns))

ds0 = xr.open_mfdataset(fns, combine='by_coords')

time = []
for t in ds0.forecast_initial_time.values:
    for h in ds0.forecast_hour.values:
        time.append(np.datetime64(t) + np.timedelta64(int(h), 'h'))
time = np.array(time)
print(time.shape)

ds0_flat = ds0.stack(time=('forecast_initial_time', 'forecast_hour'))
ds0_flat = ds0_flat.assign_coords(time=("time", time))
ds0_flat

(27048,)


/glade/derecho/scratch/khirata/tmp/ipykernel_12710/3566190605.py:32: FutureWarning: updating coordinate 'time' with a PandasMultiIndex would leave the multi-index level coordinates ['forecast_initial_time', 'forecast_hour'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['time', 'forecast_initial_time', 'forecast_hour'])` before assigning new coordinate values.
  ds0_flat = ds0_flat.assign_coords(time=("time", time))


<xarray.Dataset> Size: 225GB
Dimensions:    (latitude: 721, longitude: 1440, time: 27048)
Coordinates:
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * time       (time) datetime64[ns] 216kB 2018-12-01T07:00:00 ... 2022-01-01...
Data variables:
    MLSPR      (latitude, longitude, time) float32 112GB dask.array<chunksize=(721, 1440, 12), meta=np.ndarray>
    utc_date   (time) int32 108kB dask.array<chunksize=(360,), meta=np.ndarray>
    MCPR       (latitude, longitude, time) float32 112GB dask.array<chunksize=(721, 1440, 12), meta=np.ndarray>
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.6.3
    CONVERSION_PLATFORM:  Linux r1i1n16 4.12.14-94.41-default #1 SMP Wed Oct ...
    CONVERSION_DATE:      Mon Sep  9 09:14:56 MDT 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Mon Sep  9 09:15:04 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.9 (Homepage = http://n...

In [29]:
ds0_flat.nbytes  / 1e6 # data size check (MB)

224658.866024

#### A function to add edge data points for cyclicity (experimental; currently not used as it takes too much memory)

In [5]:
from cartopy.util import add_cyclic_point

def xr_add_cyclic_point(ds, lonname='longitude'):
    """Add a cyclic column to the longitude dim for all DataArrays in a Dataset."""

    # Prepare new variables and coordinates
    new_vars = {}
    for varname, da in ds.data_vars.items():
        if lonname in da.dims:
            lon_idx = da.dims.index(lonname)
            wrap_data, wrap_lon = add_cyclic_point(da.values, coord=ds[lonname], axis=lon_idx)
            coords = {n: c for n, c in da.coords.items() if n != lonname}
            coords[lonname] = wrap_lon
            new_vars[varname] = xr.DataArray(data=wrap_data, coords=coords, dims=da.dims, attrs=da.attrs)
        else:
            new_vars[varname] = da

    # Update coordinates
    new_coords = dict(ds.coords)
    if lonname in new_coords:
        new_coords[lonname] = wrap_lon

    return xr.Dataset(new_vars, coords=new_coords, attrs=ds.attrs)


NetCDF file save location:

In [30]:
dir_scratch = '/glade/derecho/scratch/khirata/'

### Compute weights for remapping from lat-lon to HEALPix grid

In [53]:
# ds1 = xr_add_cyclic_point(ds0_flat)
# ds = ds1.stack(xy=("longitude", "latitude"))

ds = ds0_flat.stack(xy=("longitude", "latitude"))

zoom = 7 #6 #9
# zoom = 6
# zoom = 5
# zoom = 4
# zoom = 3
# zoom = 2
nside = hp.order2nside(zoom)
npix = hp.nside2npix(nside)

hp_lon, hp_lat = hp.pix2ang(nside=nside, ipix=np.arange(npix), lonlat=True, nest=True)
hp_lon = ((hp_lon + 180) % 360 )
# Lucas had a 180 deg shift that should not be used for X-SHiELD
#hp_lon = (hp_lon + 180) % 360 - 180  # [-180, 180)
hp_lon += 360 / (4 * nside) / 4  # shift quarter-width                                                                    

# compute weights or load precomputed ones
# use_precomputed_weights = True
use_precomputed_weights = False
if use_precomputed_weights:
    weight_fn = dir_scratch + 'healpix_weights_lv%d.nc' % zoom
    weights = xr.open_dataset(weight_fn)
else:
    weights = egr.compute_weights_delaunay((ds.longitude, ds.latitude), (hp_lon, hp_lat))
    # You can also save the calculated weights for future use
    weights.to_netcdf(dir_scratch + 'healpix_weights_lv%d.nc' % zoom)


### Do the remapping

In [54]:
ds_remap = xr.apply_ufunc(
    egr.apply_weights,
    ds,
    kwargs=weights,
    # keep_attrs=True,
    input_core_dims=[["xy"]],
    output_core_dims=[["cell"]],
    on_missing_core_dim="copy",
    output_dtypes=["f4"],
    vectorize=True,
    dask="parallelized",
    dask_gufunc_kwargs={
        "output_sizes": {"cell": npix},
    },
)

### Reconstruct the UXArray Dataset (for any subsequent analyses)

In [55]:
uxds = ux.UxDataset.from_xarray(ds_remap, ux.Grid.from_healpix(zoom, pixels_only=True, nest=True))

In [56]:
uxds.nbytes / 1e6 # data size check (MB)

42543.150048

In [57]:
uxda_pr_era5_pre = uxds['MLSPR'] + uxds['MCPR'] # combine two types of precipitation rate
uxda_pr_era5_pre

<xarray.UxDataArray (time: 27048, n_face: 196608)> Size: 21GB
dask.array<add, shape=(27048, 196608), dtype=float32, chunksize=(12, 196608), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 216kB 2018-12-01T07:00:00 ... 2022-01-01T0...
Dimensions without coordinates: n_face

In [58]:
dirname = 'era5_zarr'

varname = 'mtotpr'

idname = 'era5_%s_%d_%d_zmlv%d' % (varname, int(start_date.split('-')[0]), int(end_date.split('-')[0]), zoom)

In [59]:
%%time
import zarr
import numcodecs
import numpy as np

# nch_t = 1024
# nch_c = 1024
# nch_t = 1
# nch_c = uxda_pr_era5_pre.sizes['n_face']
nch_t = 256 * 4**(7 - zoom)
nch_c = uxda_pr_era5_pre.sizes['n_face']


def get_dtype(da):
    if np.issubdtype(da.dtype, np.floating):
        return "float32"
    else:
        return da.dtype
        
def get_chunks(dimensions):
    chunks = {
        "time": nch_t,
        "cell": nch_c,
    }

    return tuple((chunks[d] for d in dimensions))

def get_compressor():
    return numcodecs.Blosc("zstd", shuffle=2)



def get_encoding(dataset):
    return {
        var: {
            "compressor": get_compressor(),
            "dtype": get_dtype(dataset[var]),
            "chunks": get_chunks(dataset[var].dims),
        }
        for var in dataset.variables
        if var not in dataset.dims
    }

store = zarr.storage.DirectoryStore(dir_scratch + '%s/%s.zarr' % (dirname, idname), dimension_separator='/')

uxda_pr_era5_pre.name = varname
uxds_pr_era5 = uxda_pr_era5_pre.swap_dims({'n_face': 'cell'}).to_dataset()

# uxds_pr_era5.to_zarr(store, encoding=get_encoding(uxds_pr_era5))
uxds_pr_era5_rechunked = uxds_pr_era5.chunk({'time': nch_t, 'cell': nch_c}) #.compute()
uxds_pr_era5_rechunked.to_zarr(store, encoding=get_encoding(uxds_pr_era5))


CPU times: user 44.4 s, sys: 8.9 s, total: 53.3 s
Wall time: 5min 13s


In [60]:
uxds_pr_era5_rechunked

<xarray.UxDataset> Size: 21GB
Dimensions:  (time: 27048, cell: 196608)
Coordinates:
  * time     (time) datetime64[ns] 216kB 2018-12-01T07:00:00 ... 2022-01-01T0...
Dimensions without coordinates: cell
Data variables:
    mtotpr   (time, cell) float32 21GB dask.array<chunksize=(256, 196608), meta=np.ndarray>

In [65]:
cluster.close()

### Testing the zarr file

In [61]:
uxds_era5 = ux.UxDataset.from_healpix(dir_scratch + '%s/%s.zarr' % (dirname, idname))

In [62]:
uxds_era5['mtotpr'].isel(time=0).plot()

:Image   [x,y]   (x_y mtotpr)

In [63]:
uxda_era5_daily = uxda_pr_era5_pre.resample(time='1D').mean()

In [64]:
uxda_era5_daily.isel(time=0).plot()

:Image   [x,y]   (x_y mtotpr)

In [ ]:
# uxda_era5_daily.to_netcdf(dir_scratch + 'era5_daily_precip_lv%d.nc' % zoom)